# CompactMix-L: chạy code và báo cáo CIFAR-10 / CIFAR-100

**Sinh viên:** ................................ · **Ngày trình bày:** 16/10/2026

Bố cục theo notebook [Training a Classifier của PyTorch](https://colab.research.google.com/github/pytorch/tutorials/blob/gh-pages/_downloads/cifar10_tutorial.ipynb): nạp dữ liệu → định nghĩa mạng/forward → loss/optimizer → train → lưu/nạp → đánh giá từng lớp. Kiến trúc và recipe ở đây là **CompactMix-L của bài làm**, không dùng mạng Net nhỏ của tutorial.

Notebook chứa đầy đủ code model và train, **không import `src/` hoặc file Python của dự án**. Mở trong VS Code/Jupyter và chọn đúng kernel Microsoft Store Python 3.13 đang dùng. Link Colab là mẫu bố cục; notebook này giữ chính sách môi trường Python hiện tại của bạn.

### Cách dùng khi trình bày

1. Chạy từ trên xuống; xem ảnh, code từng block, bảng shape và kiểm tra ngân sách.
2. Mặc định `MODE="demo"`: dùng **512 ảnh train + 256 validation mỗi dataset**, chạy **một epoch mới mỗi lần gọi**; tổng lịch đặt trước là 2 epoch. Chạy lại cell train để minh họa resume. Accuracy demo không phải điểm số trong PDF.
3. Các cell cuối đọc và kiểm chứng **kết quả đã lưu của 600 epoch/45k**, vẽ lại learning curves, confusion matrix và chỉ số từng lớp. Không cần train lại 600 epoch để trình bày các kết quả đã đo.
4. `MODE="report_only"` bỏ qua train mới. `MODE="full"` dùng đầy đủ 45k/5k, lịch 600 epoch; chỉ chọn khi thực sự muốn train dài.

Checkpoint lượt mới nằm riêng trong `experiments/notebook_presentation/`. Mỗi run chỉ có `best.pt` và `last.pt`; log train chỉ ghi sau epoch hoàn chỉnh. Không ghi đè checkpoint nguồn của báo cáo.

## 1. Đề bài và phạm vi kết quả

Theo `Final exam.docx`: model L có **≤6.000.000 tham số học được**, **<1.000.000.000 FLOPs**, train/test CIFAR-10 và CIFAR-100, so sánh SGD/Adam và nhiều LR, mô tả learning settings trong PDF.

| Chỉ số trong PDF, snapshot 14/10/2026 | CIFAR-10 | CIFAR-100 |
|---|---:|---:|
| Epoch selection đã hoàn thành | 600 | 600 |
| Best epoch / weights | 567 / EMA | 399 / EMA |
| Best validation (5k) | 97,56% | 83,36% |
| Test best train 45k (10k ảnh) | 97,18% | 82,98% |
| Tham số | 5.248.172 | 5.271.302 |
| FLOPs/ảnh theo profiler bài làm | 842.406.944 | 842.453.114 |

Đây là **kết quả cũ đã đo của best train 45k**, không phải kết quả của lượt demo hay retrain 50k. Snapshot PDF chưa có final-test của retrain 50k; so sánh full-data Adam/nhiều LR, multi-seed và ablation còn thiếu. Các cell đọc evidence bên dưới sẽ kiểm chứng bảng này bằng hash checkpoint và số đếm confusion matrix.

## 2. Import, kiểm tra đúng Python/CUDA và xác định thư mục

Cell này chỉ dùng các thư viện đã có trong Python được chọn; không tạo môi trường hoặc cài thư viện vào nơi khác.

In [ ]:
import copy
import csv
from contextlib import contextmanager
from datetime import datetime, timezone
import hashlib
import json
import logging
import math
import os
from pathlib import Path
import platform
import random
import re
import subprocess
import sys
import time

def ensure_python():
    selected = r"C:\Users\anhna\AppData\Local\Microsoft\WindowsApps\python3.13.exe"
    resolved = r"C:\Users\anhna\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe"
    norm = lambda value: os.path.normcase(os.path.abspath(value))
    if (norm(sys.executable) not in {norm(selected), norm(resolved)}
            or norm(sys.prefix) != norm(sys.base_prefix)
            or sys.version_info[:2] != (3, 13)):
        raise RuntimeError("Chỉ dùng Microsoft Store Python 3.13 đã chọn: " + selected)

ensure_python()
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import datasets, transforms
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython import get_ipython
from IPython.display import display, Markdown

get_ipython().run_line_magic("matplotlib", "inline")
torch.set_num_threads(min(4, os.cpu_count() or 1))
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_colwidth", 65)
plt.rcParams.update({"figure.dpi": 115, "axes.spines.top": False, "axes.spines.right": False})
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd,*cwd.parents) if (p/"datasets").is_dir()),
            cwd.parent if cwd.name == "report" else cwd)
REPORT_DIR = ROOT/"report"
DATA_ROOT = ROOT/"datasets"
OUTPUT_ROOT = ROOT/"experiments/notebook_presentation"
CHECKPOINT_VERSION = 1
REQUEST_STOP = False
NOTEBOOK_TRAINING_CELLS = {}

def _remember_cell(name):
    # Lưu source cell THỰC SỰ đã chạy, kể cả sửa trong IDE chưa Save.
    # Chạy lại cell định nghĩa sẽ thay source của chính cell đó.
    shell = get_ipython()
    source = shell.history_manager.input_hist_raw[-1]
    if not source.strip():
        raise RuntimeError("Không đọc được source cell đang chạy; chạy notebook bằng IPython/Jupyter.")
    NOTEBOOK_TRAINING_CELLS[name] = source.replace("\r\n", "\n")

_remember_cell("bootstrap")
display(pd.DataFrame([{
    "Python": platform.python_version(), "Executable": sys.executable,
    "Base environment": sys.prefix == sys.base_prefix,
    "PyTorch": torch.__version__, "Torchvision": torchvision.__version__,
    "CUDA build": torch.version.cuda,
    "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "torch location": torch.__file__, "numpy location": np.__file__,
}]))
print("Dữ liệu:", DATA_ROOT, "\nLượt chạy mới:", OUTPUT_ROOT)

## 3. Cấu hình chạy và recipe

`DEMO_STOP_AFTER=1` chỉ tạm dừng sau một epoch mới; **không đổi lịch tổng `DEMO_TOTAL_EPOCHS=2` giữa hai lần resume**. Chạy lại cell train để tiếp tục. Nếu sửa model/recipe/source, đổi `RUN_NAME` để tạo thí nghiệm mới.

Notebook dùng `workers=0` để các class Dataset định nghĩa trong cell chạy ổn định với Jupyter trên Windows. PDF đã train bằng `workers=2`; cách đặt worker không làm thay đổi kiến trúc. Các giá trị báo cáo cũ vẫn đọc từ config đã lưu.

In [ ]:
TRAIN_SETTINGS = {
    "epochs":600, "batch_size":128, "workers":2, "seed":42, "split_seed":2026,
    "optimizer":"sgd", "lr":0.1, "weight_decay":0.0005, "momentum":0.9,
    "warmup":5, "scheduler":"cosine", "label_smoothing":0.05,
    "mixup":0.2, "cutmix":1.0, "mix_probability":0.75, "augmentation":"autoaugment",
    "random_erasing":0.0, "ema_decay":0.999, "grad_clip":5.0, "amp":True,
    "deterministic":True, "tensorboard":False,
}

STATS = {
    "cifar10":((.4914,.4822,.4465),(.2470,.2435,.2616)),
    "cifar100":((.5071,.4867,.4408),(.2675,.2565,.2761)),
}


# Có thể sửa các biến này ngay trong cell, rồi chạy lại.
MODE = globals().get("NB_MODE", "demo")           # demo | report_only | full
DATASETS = ("cifar10", "cifar100")
RUN_NAME = globals().get("NB_RUN_NAME", "bao_cao_20261016")
DEMO_TOTAL_EPOCHS = 2
DEMO_STOP_AFTER = 1
DEMO_TRAIN_IMAGES = 512
DEMO_VAL_IMAGES = 256
FULL_EPOCHS = 600
DOWNLOAD = False                                # dùng datasets/ hiện có
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SYNTHETIC_CHECK = bool(globals().get("NB_SMOKE", False))  # chỉ dành cho smoke kỹ thuật
RUN_MATRIX = False                              # bật riêng khi muốn train nhiều LR
RUN_FINAL = False                               # bật khi selection full đã hoàn tất
RECHECK_ARCHIVED_TEST = False                    # mặc định đọc điểm test đã lưu
TRAIN_SETTINGS.update(workers=0, tensorboard=False)
if MODE not in {"demo", "report_only", "full"}:
    raise ValueError("MODE phải là demo, report_only hoặc full")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", RUN_NAME):
    raise ValueError("RUN_NAME chỉ dùng chữ, số, dấu chấm/gạch dưới/gạch ngang")
display(pd.DataFrame([TRAIN_SETTINGS]).T.rename(columns={0:"Recipe mặc định"}))
print("Chế độ:", MODE, "| datasets:", DATASETS, "| device:", DEVICE)

## 4. Dữ liệu, stratified split, augmentation và xem ảnh

Mỗi dataset có 50k train, 10k test, ảnh RGB 32×32. Tách stratified 45k/5k từ official train bằng seed 2026; lưu indices và SHA256. **Phần lựa chọn model chỉ mở `train=True`.** Train dùng RandomCrop/Flip/AutoAugment + Normalize; validation chỉ ToTensor/Normalize.

Với batch size 128: `[N,3,32,32]`; N có thể nhỏ hơn 128 ở batch cuối. Demo cắt một subset xác định để chạy nhanh và được gắn `subset=True` trong checkpoint/result.

In [ ]:
_remember_cell("data-definitions")

def stratified_split(targets, classes, seed):
    rng = np.random.default_rng(seed)
    targets = np.asarray(targets)
    train, val = [], []
    for label in range(classes):
        indices = np.flatnonzero(targets == label)
        if len(indices) < 2:
            raise ValueError('Each class needs at least two samples')
        rng.shuffle(indices)
        n = max(1, len(indices) // 10)
        val.extend(indices[:n].tolist())
        train.extend(indices[n:].tolist())
    return sorted(train), sorted(val)

def prepare_data(config):
    classes = 10 if config['dataset'] == 'cifar10' else 100
    if config['smoke']:
        source = datasets.FakeData(size=48, image_size=(3, 32, 32), num_classes=classes)
        training, validation = list(range(32)), list(range(32, 48))
    else:
        constructor = datasets.CIFAR10 if classes == 10 else datasets.CIFAR100
        source = constructor(root=config['data_root'], train=True, download=config['download'])
        training, validation = stratified_split(source.targets, classes, config['split_seed'])
    if config['phase'] == 'final':
        training = sorted(training + validation)
        validation = []
    if config['limit_train']:
        training = training[:config['limit_train']]
    if config['limit_val']:
        validation = validation[:config['limit_val']]
    payload = {'training': training, 'validation': validation}
    return source, {**payload, 'classes': classes,
                    'sha256': hashlib.sha256(json.dumps(payload).encode()).hexdigest(),
                    'synthetic': config['smoke'],
                    'subset': bool(config['limit_train'] or config['limit_val'])}

@contextmanager
def sample_rng(seed):
    python_state, numpy_state = random.getstate(), np.random.get_state()
    with torch.random.fork_rng(devices=[]):
        random.seed(seed)
        np.random.seed(seed % 2**32)
        torch.random.default_generator.manual_seed(seed)
        try:
            yield
        finally:
            random.setstate(python_state)
            np.random.set_state(numpy_state)

def image_transforms(config, training):
    mean, std = STATS[config['dataset']]
    ops = []
    if training:
        ops = [transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
               transforms.RandomHorizontalFlip()]
        kind = config['augmentation']
        if kind == 'autoaugment':
            ops.append(transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10))
        elif kind == 'randaugment':
            ops.append(transforms.RandAugment(num_ops=2, magnitude=9))
        elif kind == 'trivialaugment':
            ops.append(transforms.TrivialAugmentWide())
        elif kind != 'basic':
            raise ValueError(f'Unknown augmentation: {kind}')
    ops += [transforms.ToTensor(), transforms.Normalize(mean, std)]
    if training and config['random_erasing']:
        ops.append(transforms.RandomErasing(p=config['random_erasing']))
    return transforms.Compose(ops)

class SeededImages(Dataset):
    def __init__(self, source, indices, transform, seed, epoch):
        self.source, self.indices, self.transform = source, indices, transform
        self.seed, self.epoch = seed, epoch

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = self.indices[position]
        seed = (self.seed * 1_000_003 + self.epoch * 97_409 + index) % (2**63-1)
        with sample_rng(seed):
            image, label = self.source[index]
            return self.transform(image), label

def make_loader(source, indices, config, epoch, training=False):
    dataset = SeededImages(source, indices, image_transforms(config, training), config['seed'], epoch)
    generator = torch.Generator().manual_seed(config['seed'] + epoch * 1_000_003)
    return DataLoader(dataset, batch_size=config['batch_size'], shuffle=training,
                      num_workers=config['workers'], generator=generator,
                      pin_memory=config['device'] == 'cuda', persistent_workers=False,
                      **({'prefetch_factor': 2} if config['workers'] else {}))

In [ ]:
def notebook_configs(mode=None, optimizer=None, lr=None):
    mode = MODE if mode is None else mode
    configs = []
    for name in DATASETS:
        cfg = copy.deepcopy(TRAIN_SETTINGS)
        cfg.update(dataset=name, model={"family":"compactmix_manual"},
                   phase="selection", final_weights="ema", device=DEVICE,
                   data_root=str(DATA_ROOT.resolve()), download=DOWNLOAD,
                   smoke=SYNTHETIC_CHECK, limit_train=0, limit_val=0)
        cfg["epochs"] = FULL_EPOCHS if mode=="full" else DEMO_TOTAL_EPOCHS
        if mode!="full":
            cfg.update(limit_train=DEMO_TRAIN_IMAGES, limit_val=DEMO_VAL_IMAGES)
        if SYNTHETIC_CHECK:
            cfg.update(limit_train=0,limit_val=0,batch_size=4)
        if optimizer:
            cfg["optimizer"] = optimizer
            if optimizer in ("adam","adamw"):
                cfg.update(lr=.001,weight_decay=.0001)
        if lr is not None:
            cfg["lr"] = lr
        label=f"{RUN_NAME}_{mode}_{name}_{cfg['optimizer']}_lr{cfg['lr']:g}_seed{cfg['seed']}"
        cfg["run_dir"] = str((OUTPUT_ROOT/label).resolve())
        configs.append(cfg)
    return configs

CONFIGS = notebook_configs()
DATA_PREVIEW = {}
for cfg in CONFIGS:
    source, split = prepare_data(cfg)
    DATA_PREVIEW[cfg["dataset"]] = (source,split)
    class_names = getattr(source,"classes",[str(i) for i in range(split["classes"])])
    fig, axes = plt.subplots(2,4,figsize=(9,4))
    for ax,index in zip(axes.flat,split["training"][:8]):
        image,label = source[index]
        ax.imshow(image); ax.set_title(class_names[label]); ax.axis("off")
    fig.suptitle(cfg["dataset"]+" — ảnh TRAIN, chưa normalize")
    plt.tight_layout(); plt.show(); plt.close(fig)
    print(cfg["dataset"], "train:",len(split["training"]),"validation:",len(split["validation"]),
          "synthetic:",split["synthetic"],"subset:",split["subset"],"split SHA:",split["sha256"])

## 5. Định nghĩa model: shape checks, DropPath và 13 block tường minh

Widths **40/112/256**, depths **2/3/8**, kích thước **32/16/8**. Mỗi block khai báo Conv/BN/SE và forward riêng để bạn tự sửa các thông số liên quan. Không tự suy ra hay vá kênh.

- `chunk(2,dim=1)`: chia kênh làm hai nhánh.
- Concat: N/H/W bằng nhau; C đầu ra là tổng C hai nhánh.
- SE multiply: feature `[N,C,H,W]`, gate đúng `[N,C,1,1]`.
- Residual ADD: hai tensor bằng nhau cả N/C/H/W.

Conv: `H_out=floor((H_in+2*pH-dH*(kH-1)-1)/sH+1)`, W tương tự. BN/SiLU/Sigmoid/DropPath giữ shape. Xem thêm [công thức shape](CONG_THUC_TINH_SHAPE.md).

In [ ]:
_remember_cell("shape-helpers")

class ShapeError(ValueError):
    """Lỗi kết nối cần người học tự sửa thông số."""

def concat_checked(a, b, where):
    if a.shape[0] != b.shape[0] or a.shape[2:] != b.shape[2:]:
        raise ShapeError(f"{where}: local={list(a.shape)}, context={list(b.shape)}. "
                         "Tự sửa kernel/stride/padding để hai nhánh cùng H,W trước cat(dim=1).")
    return torch.cat((a, b), dim=1)

def multiply_checked(feature, gate, where):
    expected = (feature.shape[0], feature.shape[1], 1, 1)
    if tuple(gate.shape) != expected:
        raise ShapeError(f"{where}: feature={list(feature.shape)}, gate={list(gate.shape)}. "
                         f"SE phải trả gate={list(expected)}; sửa Conv cuối của SE.")
    return feature * gate

def residual_checked(x, residual, where):
    if x.shape != residual.shape:
        raise ShapeError(f"{where}: skip={list(x.shape)}, residual={list(residual.shape)}. "
                         "Tự sửa projection/stride/skip để hai tensor bằng shape trước cộng; "
                         "không dùng broadcast để che lỗi.")
    return x + residual

class DropPath(nn.Module):
    def __init__(self, probability):
        super().__init__()
        if not 0 <= probability < 1:
            raise ValueError("DropPath probability phải thuộc [0,1).")
        self.probability = probability

    def forward(self, x):
        if not self.training or self.probability == 0:
            return x
        keep = 1-self.probability
        mask = torch.rand((x.shape[0], 1, 1, 1), device=x.device) < keep
        return x * mask.to(x.dtype) / keep

class ManualBlock(nn.Module):
    # Chỉ dùng làm nhãn cho profiler; không tạo/sửa Conv hoặc số kênh.
    pass

### Block01 — Stage 1, input/output `[N,40,32,32]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block01")

class Block01(ManualBlock):
    """Stage 1; input/output [N,40,32,32]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=40, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=40, out_channels=80, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=80, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=40, out_channels=40, kernel_size=3, stride=1, padding=1, dilation=1, groups=40, bias=False)
        self.context = nn.Conv2d(in_channels=40, out_channels=40, kernel_size=5, stride=1, padding=2, dilation=1, groups=40, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=80, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=80, out_channels=80, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=80, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=80, out_channels=8, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=8, out_channels=80, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=80, out_channels=40, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.0)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block01.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block01.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block01.ADD")

### Block02 — Stage 1, input/output `[N,40,32,32]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block02")

class Block02(ManualBlock):
    """Stage 1; input/output [N,40,32,32]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=40, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=40, out_channels=80, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=80, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=40, out_channels=40, kernel_size=3, stride=1, padding=1, dilation=1, groups=40, bias=False)
        self.context = nn.Conv2d(in_channels=40, out_channels=40, kernel_size=5, stride=1, padding=2, dilation=1, groups=40, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=80, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=80, out_channels=80, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=80, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=80, out_channels=8, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=8, out_channels=80, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=80, out_channels=40, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.008333333333333333)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block02.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block02.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block02.ADD")

### Block03 — Stage 2, input/output `[N,112,16,16]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block03")

class Block03(ManualBlock):
    """Stage 2; input/output [N,112,16,16]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=112, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=112, out_channels=224, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=224, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=112, out_channels=112, kernel_size=3, stride=1, padding=1, dilation=1, groups=112, bias=False)
        self.context = nn.Conv2d(in_channels=112, out_channels=112, kernel_size=5, stride=1, padding=2, dilation=1, groups=112, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=224, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=224, out_channels=224, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=224, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=224, out_channels=14, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=14, out_channels=224, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=224, out_channels=112, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.016666666666666666)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block03.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block03.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block03.ADD")

### Block04 — Stage 2, input/output `[N,112,16,16]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block04")

class Block04(ManualBlock):
    """Stage 2; input/output [N,112,16,16]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=112, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=112, out_channels=224, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=224, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=112, out_channels=112, kernel_size=3, stride=1, padding=1, dilation=1, groups=112, bias=False)
        self.context = nn.Conv2d(in_channels=112, out_channels=112, kernel_size=5, stride=1, padding=2, dilation=1, groups=112, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=224, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=224, out_channels=224, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=224, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=224, out_channels=14, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=14, out_channels=224, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=224, out_channels=112, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.025000000000000005)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block04.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block04.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block04.ADD")

### Block05 — Stage 2, input/output `[N,112,16,16]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block05")

class Block05(ManualBlock):
    """Stage 2; input/output [N,112,16,16]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=112, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=112, out_channels=224, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=224, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=112, out_channels=112, kernel_size=3, stride=1, padding=1, dilation=1, groups=112, bias=False)
        self.context = nn.Conv2d(in_channels=112, out_channels=112, kernel_size=5, stride=1, padding=2, dilation=1, groups=112, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=224, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=224, out_channels=224, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=224, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=224, out_channels=14, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=14, out_channels=224, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=224, out_channels=112, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.03333333333333333)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block05.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block05.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block05.ADD")

### Block06 — Stage 3, input/output `[N,256,8,8]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block06")

class Block06(ManualBlock):
    """Stage 3; input/output [N,256,8,8]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=256, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1, dilation=1, groups=256, bias=False)
        self.context = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=5, stride=1, padding=2, dilation=1, groups=256, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=512, out_channels=32, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=32, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.041666666666666664)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block06.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block06.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block06.ADD")

### Block07 — Stage 3, input/output `[N,256,8,8]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block07")

class Block07(ManualBlock):
    """Stage 3; input/output [N,256,8,8]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=256, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1, dilation=1, groups=256, bias=False)
        self.context = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=5, stride=1, padding=2, dilation=1, groups=256, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=512, out_channels=32, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=32, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.05000000000000001)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block07.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block07.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block07.ADD")

### Block08 — Stage 3, input/output `[N,256,8,8]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block08")

class Block08(ManualBlock):
    """Stage 3; input/output [N,256,8,8]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=256, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1, dilation=1, groups=256, bias=False)
        self.context = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=5, stride=1, padding=2, dilation=1, groups=256, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=512, out_channels=32, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=32, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.05833333333333334)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block08.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block08.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block08.ADD")

### Block09 — Stage 3, input/output `[N,256,8,8]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block09")

class Block09(ManualBlock):
    """Stage 3; input/output [N,256,8,8]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=256, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1, dilation=1, groups=256, bias=False)
        self.context = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=5, stride=1, padding=2, dilation=1, groups=256, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=512, out_channels=32, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=32, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.06666666666666667)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block09.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block09.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block09.ADD")

### Block10 — Stage 3, input/output `[N,256,8,8]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block10")

class Block10(ManualBlock):
    """Stage 3; input/output [N,256,8,8]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=256, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1, dilation=1, groups=256, bias=False)
        self.context = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=5, stride=1, padding=2, dilation=1, groups=256, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=512, out_channels=32, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=32, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.075)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block10.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block10.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block10.ADD")

### Block11 — Stage 3, input/output `[N,256,8,8]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block11")

class Block11(ManualBlock):
    """Stage 3; input/output [N,256,8,8]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=256, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1, dilation=1, groups=256, bias=False)
        self.context = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=5, stride=1, padding=2, dilation=1, groups=256, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=512, out_channels=32, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=32, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.08333333333333333)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block11.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block11.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block11.ADD")

### Block12 — Stage 3, input/output `[N,256,8,8]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block12")

class Block12(ManualBlock):
    """Stage 3; input/output [N,256,8,8]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=256, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1, dilation=1, groups=256, bias=False)
        self.context = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=5, stride=1, padding=2, dilation=1, groups=256, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=512, out_channels=32, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=32, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.09166666666666667)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block12.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block12.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block12.ADD")

### Block13 — Stage 3, input/output `[N,256,8,8]`

Conv expand → hai nhánh depthwise 3×3/5×5 → concat → grouped Conv → SE → projection → DropPath → ADD.

In [ ]:
_remember_cell("block13")

class Block13(ManualBlock):
    """Stage 3; input/output [N,256,8,8]. Sửa từng số bên dưới bằng tay."""
    def __init__(self):
        super().__init__()
        self.bn_pre = nn.BatchNorm2d(num_features=256, eps=1e-5, momentum=0.1)
        self.act_pre = nn.SiLU()
        self.expand = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.bn_expand = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_expand = nn.SiLU()
        self.local = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1, dilation=1, groups=256, bias=False)
        self.context = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=5, stride=1, padding=2, dilation=1, groups=256, bias=False)
        self.bn_mix = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_mix = nn.SiLU()
        self.grouped = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, dilation=1, groups=8, bias=False)
        self.bn_gate = nn.BatchNorm2d(num_features=512, eps=1e-5, momentum=0.1)
        self.act_gate = nn.SiLU()
        self.se_pool = nn.AdaptiveAvgPool2d(output_size=1)
        self.se_reduce = nn.Conv2d(in_channels=512, out_channels=32, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_act = nn.SiLU()
        self.se_expand = nn.Conv2d(in_channels=32, out_channels=512, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=True)
        self.se_sigmoid = nn.Sigmoid()
        self.project = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=1, stride=1, padding=0, dilation=1, groups=1, bias=False)
        self.drop = DropPath(probability=0.10000000000000002)

    def forward(self, x):
        y = self.bn_pre(x)
        y = self.act_pre(y)
        y = self.expand(y)
        y = self.bn_expand(y)
        y = self.act_expand(y)
        a, b = y.chunk(2, dim=1)
        a = self.local(a)
        b = self.context(b)
        y = concat_checked(a, b, "Block13.concat")
        y = self.bn_mix(y)
        y = self.act_mix(y)
        y = self.grouped(y)
        y = self.bn_gate(y)
        y = self.act_gate(y)
        feature = y
        gate = self.se_pool(feature)
        gate = self.se_reduce(gate)
        gate = self.se_act(gate)
        gate = self.se_expand(gate)
        gate = self.se_sigmoid(gate)
        self._gate_elements = feature.numel()  # Chỉ để đếm FLOPs.
        y = multiply_checked(feature, gate, "Block13.SE")
        y = self.project(y)
        y = self.drop(y)
        return residual_checked(x, y, "Block13.ADD")

### Ghép stem → stage → transition → head

`self.stem` là Conv 3×3 đầu tiên 3→40. Trong `forward`, stem chạy trước Block01; head trả logits, không thêm softmax trước loss.

In [ ]:
_remember_cell("whole-model")

class CompactMixManual(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Conv2d(3, 40, kernel_size=3, stride=1, padding=1, groups=1, bias=False)
        self.block01 = Block01()
        self.block02 = Block02()
        self.t1_bn = nn.BatchNorm2d(40, eps=1e-5, momentum=0.1)
        self.t1_act = nn.SiLU()
        self.t1_pool = nn.AvgPool2d(kernel_size=2, stride=2)
        self.t1_conv = nn.Conv2d(40, 112, kernel_size=1, stride=1, padding=0, bias=False)
        self.block03 = Block03()
        self.block04 = Block04()
        self.block05 = Block05()
        self.t2_bn = nn.BatchNorm2d(112, eps=1e-5, momentum=0.1)
        self.t2_act = nn.SiLU()
        self.t2_pool = nn.AvgPool2d(kernel_size=2, stride=2)
        self.t2_conv = nn.Conv2d(112, 256, kernel_size=1, stride=1, padding=0, bias=False)
        self.block06 = Block06()
        self.block07 = Block07()
        self.block08 = Block08()
        self.block09 = Block09()
        self.block10 = Block10()
        self.block11 = Block11()
        self.block12 = Block12()
        self.block13 = Block13()
        self.head_bn = nn.BatchNorm2d(256, eps=1e-5, momentum=0.1)
        self.head_act = nn.SiLU()
        self.head_pool = nn.AdaptiveAvgPool2d(1)
        self.head_dropout = nn.Dropout(p=0.0)
        # Chỉ số lớp theo dataset; in_features=256 phải tự sửa nếu đổi Stage 3.
        self.classifier = nn.Linear(in_features=256, out_features=num_classes, bias=True)
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.02)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.block01(x)
        x = self.block02(x)
        x = self.t1_bn(x)
        x = self.t1_act(x)
        x = self.t1_pool(x)
        x = self.t1_conv(x)
        x = self.block03(x)
        x = self.block04(x)
        x = self.block05(x)
        x = self.t2_bn(x)
        x = self.t2_act(x)
        x = self.t2_pool(x)
        x = self.t2_conv(x)
        x = self.block06(x)
        x = self.block07(x)
        x = self.block08(x)
        x = self.block09(x)
        x = self.block10(x)
        x = self.block11(x)
        x = self.block12(x)
        x = self.block13(x)
        x = self.head_bn(x)
        x = self.head_act(x)
        x = self.head_pool(x)
        x = x.flatten(1)
        x = self.head_dropout(x)
        return self.classifier(x)  # logits, không thêm softmax.

def make_model(num_classes):
    return CompactMixManual(num_classes)

## 6. Đếm tham số, MACs, FLOPs và kiểm tra forward/backward

- Conv: `P=C_out*(C_in/groups)*kH*kW + bias`; `MACs=H_out*W_out*C_out*(C_in/groups)*kH*kW` cho N=1.
- Linear: `P=F_in*F_out+F_out` khi có bias; `MACs=F_in*F_out` cho N=1.
- BN affine: `P=2*C`; running stats là buffer.
- Convention của PDF: **1 MAC=2 FLOPs**, cộng bias/BN/SiLU/Sigmoid/pooling/SE multiply/residual ADD; không đếm memory movement split/cat/reshape. Đây là ước lượng inference/ảnh, không phải latency hay training FLOPs.

Cell dưới thực chạy tensor để lấy shape. Nếu sửa layer không khớp, sẽ báo đúng lớp nối sai. Model dùng để kiểm tra là model tạm, không dùng để khởi tạo train sau khi đã cập nhật BN bởi backward thử.

In [ ]:
_remember_cell("profile-definitions")

@contextmanager
def input_checks(model):
    handles = []
    def check(name):
        def pre(module, args):
            x = args[0]
            expected = None
            actual = None
            if isinstance(module, nn.Conv2d):
                expected = module.in_channels
                actual = x.shape[1] if x.ndim == 4 else None
            elif isinstance(module, nn.BatchNorm2d):
                expected = module.num_features
                actual = x.shape[1] if x.ndim == 4 else None
            elif isinstance(module, nn.Linear):
                expected, actual = module.in_features, x.shape[-1]
            if expected is not None and actual != expected:
                raise ShapeError(f"{name}: {type(module).__name__} cần {expected} kênh/features "
                                 f"nhưng nhận {actual}, input={list(x.shape)}. "
                                 "Tự sửa output lớp trước hoặc input/BN/Linear của lớp này.")
        return pre
    try:
        for name, module in model.named_modules():
            if not list(module.children()):
                handles.append(module.register_forward_pre_hook(check(name)))
        yield
    finally:
        for handle in handles:
            handle.remove()

def inspect_model(model, num_classes, backward=True):
    """Thực chạy tensor; báo lỗi, không reshape/đệm/sửa kênh tự động."""
    stats = {"parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
             "flops": 0, "macs": 0, "layers": []}
    handles = []
    mode = model.training
    def capture(name):
        def hook(module, inputs, output):
            n, operations = output.numel(), 0
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                cost = module.in_features if isinstance(module, nn.Linear) else module.in_channels//module.groups*math.prod(module.kernel_size)
                stats["macs"] += n*cost
                operations = 2*n*cost + (n if module.bias is not None else 0)
            elif isinstance(module, nn.BatchNorm2d):
                operations = n*4
            elif isinstance(module, (nn.GroupNorm, nn.LayerNorm)):
                operations = n*8
            elif isinstance(module, (nn.SiLU, nn.ReLU, nn.GELU, nn.Sigmoid, nn.LeakyReLU)):
                operations = n*(6 if isinstance(module,nn.SiLU) else 8 if isinstance(module,nn.GELU)
                                else 4 if isinstance(module,nn.Sigmoid) else 2 if isinstance(module,nn.LeakyReLU) else 1)
            elif isinstance(module, (nn.AvgPool2d,nn.MaxPool2d,nn.AdaptiveAvgPool2d,nn.AdaptiveMaxPool2d)):
                operations = inputs[0].numel()+n
            elif isinstance(module, ManualBlock):
                operations = n+module._gate_elements  # ADD skip + MUL SE.
            elif not list(module.children()) and not isinstance(module,(DropPath,nn.Dropout,nn.Identity,nn.Flatten)):
                raise ShapeError(f"Profiler chưa hỗ trợ {name}: {type(module).__name__}; cần bổ sung cách đếm FLOPs.")
            stats["flops"] += operations
            if not list(module.children()):
                stats["layers"].append({"name":name,"type":type(module).__name__,
                    "input":list(inputs[0].shape),"output":list(output.shape),
                    "parameters":sum(p.numel() for p in module.parameters())})
        return hook
    try:
        model.cpu().eval()
        for name,module in model.named_modules():
            handles.append(module.register_forward_hook(capture(name)))
        with input_checks(model), torch.no_grad():
            output = model(torch.zeros(1,3,32,32))
        if list(output.shape) != [1,num_classes] or not torch.isfinite(output).all():
            raise ShapeError(f"Output phải là [1,{num_classes}] logits hữu hạn, nhận {list(output.shape)}.")
    finally:
        for handle in handles:
            handle.remove()
        model.train(mode)
    if stats["parameters"] > 6_000_000 or stats["flops"] >= 1_000_000_000:
        raise ShapeError(f"Vượt budget đề: P={stats['parameters']:,}, FLOPs={stats['flops']:,}; "
                         "cần P<=6M và FLOPs<1G.")
    if backward:
        model.train()
        with input_checks(model):
            logits = model(torch.randn(2,3,32,32))
            if logits.shape != (2,num_classes):
                raise ShapeError("Output train/backward không khớp số nhãn.")
            F.cross_entropy(logits,torch.tensor([0,num_classes-1])).backward()
        if not any(p.grad is not None for p in model.parameters()):
            raise ShapeError("Không có gradient; forward đã cắt graph.")
        if not all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None):
            raise ShapeError("Gradient không hữu hạn.")
        model.zero_grad(set_to_none=True)
        model.train(mode)
    return stats

In [ ]:
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
PROFILES = {}
profile_rows=[]
for name in DATASETS:
    classes=10 if name=="cifar10" else 100
    check_model=make_model(classes)
    profile=inspect_model(check_model,classes,backward=True)
    PROFILES[name]=profile
    profile_rows.append({"dataset":name,"parameters":profile["parameters"],
        "MACs/ảnh":profile["macs"],"FLOPs/ảnh":profile["flops"],"GFLOPs":profile["flops"]/1e9,
        "<=6M":profile["parameters"]<=6_000_000,"<1G":profile["flops"]<1e9,
        "forward/backward":"OK","weight FP32 MiB":sum(p.numel()*p.element_size() for p in check_model.parameters())/2**20})
    layers=pd.DataFrame(profile["layers"])
    modules=dict(check_model.named_modules())
    layers["settings"]=[str(modules[n]) for n in layers["name"]]
    layers.to_csv(OUTPUT_ROOT/f"{name}_layers.csv",index=False,encoding="utf-8-sig")
    display(Markdown(f"**{name}: đủ {len(layers)} leaf layers — shape đo bằng forward N=1**"))
    with pd.option_context("display.max_rows",300):
        display(layers)
    summary=[]
    for prefix in ("stem","block01","block02","t1","block03","block04","block05","t2",
                   *[f"block{i:02}" for i in range(6,14)],"head","classifier"):
        selected=[(n,m) for n,m in check_model.named_children() if n==prefix or n.startswith(prefix+"_")]
        summary.append({"khối":prefix,"parameters":sum(p.numel() for _,m in selected for p in m.parameters())})
    assert sum(row["parameters"] for row in summary)==profile["parameters"]
    display(pd.DataFrame(summary))
    del check_model
PROFILE_TABLE=pd.DataFrame(profile_rows)
display(PROFILE_TABLE)
PROFILE_TABLE.to_csv(OUTPUT_ROOT/"model_profiles.csv",index=False)

### Sơ đồ luồng đi của model và MixBlock

Hình được vẽ từ các khối của notebook để dùng lúc trình bày. Mỗi Conv/BN chi tiết có trong bảng layer phía trên.

In [ ]:
def draw_flow(labels,title,filename):
    fig,ax=plt.subplots(figsize=(11,max(4,len(labels)*.55)))
    ax.set_xlim(0,10); ax.set_ylim(0,len(labels)*1.2); ax.axis("off")
    for i,label in enumerate(labels):
        y=(len(labels)-i-1)*1.2+.2
        ax.add_patch(FancyBboxPatch((1,y),8,.8,boxstyle="round,pad=.07",facecolor="#e8f2ff",edgecolor="#2d5e91"))
        ax.text(5,y+.4,label,ha="center",va="center",fontsize=10)
        if i<len(labels)-1:
            ax.annotate("",xy=(5,y-.25),xytext=(5,y),arrowprops={"arrowstyle":"->","color":"#2d5e91"})
    ax.set_title(title,fontsize=15,pad=15)
    fig.tight_layout(); fig.savefig(OUTPUT_ROOT/filename,bbox_inches="tight")
    plt.show(); plt.close(fig)

draw_flow([
    "Input RGB [N,3,32,32]",
    "Stem Conv3×3: 3→40, s=1,p=1 → [N,40,32,32]",
    "Stage 1: Block01–02, C=40 → [N,40,32,32]",
    "BN → SiLU → AvgPool2×2 → Conv1×1 40→112 → [N,112,16,16]",
    "Stage 2: Block03–05, C=112 → [N,112,16,16]",
    "BN → SiLU → AvgPool2×2 → Conv1×1 112→256 → [N,256,8,8]",
    "Stage 3: Block06–13, C=256 → [N,256,8,8]",
    "Head BN → SiLU → GlobalAvgPool → [N,256,1,1]",
    "Flatten → Dropout(p=0) → [N,256]",
    "Linear 256→K → logits [N,K], K=10 hoặc 100",
],"CompactMix-L — cấu hình gốc", "architecture.png")
draw_flow([
    "x [N,C,H,W] — giữ một bản cho đường skip",
    "BN(C) → SiLU → Conv1×1 C→2C → BN(2C) → SiLU",
    "chunk(2,dim=1) → hai nhánh [N,C,H,W]",
    "Nhánh A: DW3×3,p=1 | Nhánh B: DW5×5,p=2",
    "Concat theo kênh → [N,2C,H,W] → BN → SiLU",
    "Grouped Conv3×3,groups=8,p=1 → BN → SiLU → feature",
    "SE: GAP → Conv 2C→D → SiLU → Conv D→2C → Sigmoid",
    "feature × gate [N,2C,1,1] → [N,2C,H,W]",
    "Projection Conv1×1 2C→C → DropPath → [N,C,H,W]",
    "ADD với skip x → [N,C,H,W]",
],"MixBlock — C=40/112/256, D=8/14/32", "mixblock.png")

## 7. Loss, optimizer, LR, Mixup/CutMix, EMA và reproducibility

Soft-label cross entropy: `L=-mean(sum(targets*log_softmax(logits),dim=1))`. Mixup/CutMix trộn nhãn; label smoothing=0,05. Train mixed accuracy có trọng số theo nhãn trộn; không phải accuracy sạch và không đồng nhất với validation accuracy.

SGD: LR=0,1, momentum=0,9, Nesterov; Adam/AdamW cũng có code. Warmup 5 epoch và cosine; EMA decay tối đa 0,999; gradient clipping=5; AMP chỉ bật khi CUDA. Các seed và RNG states đều lưu trong last.pt để resume.

In [ ]:
_remember_cell("recipe-definitions")

def make_optimizer(model, config):
    decay, no_decay = [], []
    for parameter in model.parameters():
        (decay if parameter.ndim > 1 else no_decay).append(parameter)
    groups = [{'params': decay, 'weight_decay': config['weight_decay']},
              {'params': no_decay, 'weight_decay': 0.}]
    kind = config['optimizer']
    if kind == 'sgd':
        return torch.optim.SGD(groups, lr=config['lr'], momentum=config['momentum'],
                               nesterov=config['momentum'] > 0)
    if kind == 'adam':
        return torch.optim.Adam(groups, lr=config['lr'], betas=(.9, .999), eps=1e-8)
    if kind == 'adamw':
        return torch.optim.AdamW(groups, lr=config['lr'], betas=(.9, .999), eps=1e-8)
    raise ValueError(f'Unknown optimizer: {kind}')

def learning_rate(config, epoch, batch, batches):
    progress = epoch + batch / batches
    warmup = min(config['warmup'], config['epochs'] / 10)
    if warmup and progress < warmup:
        return config['lr'] * (.1 + .9 * progress / warmup)
    fraction = min(1., (progress-warmup) / max(1., config['epochs']-warmup))
    if config['scheduler'] == 'cosine':
        factor = .001 + .999 * (1 + math.cos(math.pi * fraction)) / 2
    elif config['scheduler'] == 'onecycle':
        factor = (1 - math.cos(math.pi * fraction / .3)) / 2 if fraction < .3 else .001 + .999 * (1 + math.cos(math.pi * (fraction-.3)/.7)) / 2
    else:
        raise ValueError('scheduler must be cosine or onecycle')
    return config['lr'] * max(.001, factor)

def mix_batch(images, labels, classes, config, epoch):
    targets = F.one_hot(labels, classes).float()
    coefficient, paired = 1., labels
    tail = min(30, config['epochs'] // 10)
    if (epoch < config['epochs'] - tail and len(labels) > 1
            and random.random() < config['mix_probability']
            and (config['mixup'] > 0 or config['cutmix'] > 0)):
        permutation = torch.randperm(len(labels), device=images.device)
        paired = labels[permutation]
        cutmix = config['cutmix'] > 0 and (config['mixup'] <= 0 or random.random() < .5)
        if cutmix:
            coefficient = float(np.random.beta(config['cutmix'], config['cutmix']))
            height, width = images.shape[-2:]
            ratio = math.sqrt(1-coefficient)
            cut_w, cut_h = int(width*ratio), int(height*ratio)
            cx, cy = random.randrange(width), random.randrange(height)
            left, right = max(0, cx-cut_w//2), min(width, cx+cut_w//2)
            top, bottom = max(0, cy-cut_h//2), min(height, cy+cut_h//2)
            images = images.clone()
            images[:, :, top:bottom, left:right] = images[permutation, :, top:bottom, left:right]
            coefficient = 1 - (right-left)*(bottom-top)/(height*width)
        else:
            coefficient = float(np.random.beta(config['mixup'], config['mixup']))
            images = coefficient*images + (1-coefficient)*images[permutation]
        targets = coefficient*targets + (1-coefficient)*targets[permutation]
    smoothing = config['label_smoothing']
    return images, targets*(1-smoothing)+smoothing/classes, paired, coefficient

@torch.no_grad()
def update_ema(ema, model, steps, target_decay):
    decay = min(target_decay, (1+steps)/(10+steps))
    current = model.state_dict()
    for name, value in ema.state_dict().items():
        if value.is_floating_point():
            value.lerp_(current[name], 1-decay)
        else:
            value.copy_(current[name])

def seed_everything(seed, deterministic):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(deterministic)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cuda.matmul.allow_tf32 = not deterministic
    torch.set_num_threads(min(4, os.cpu_count() or 1))

def capture_rng():
    return {'python': random.getstate(), 'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}

def restore_rng(states):
    random.setstate(states['python'])
    np.random.set_state(states['numpy'])
    torch.set_rng_state(states['torch'].cpu())
    if states['cuda'] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in states['cuda']])

In [ ]:
cfg=CONFIGS[0]
seed_everything(cfg["seed"],cfg["deterministic"])
source,split=DATA_PREVIEW[cfg["dataset"]]
demo_images,demo_labels=next(iter(make_loader(source,split["training"][:8],cfg,0,training=True)))
demo_images,soft_targets,paired,coefficient=mix_batch(demo_images,demo_labels,split["classes"],cfg,0)
demo_logits=torch.randn(len(demo_labels),split["classes"])
demo_loss=-(soft_targets*F.log_softmax(demo_logits,dim=1)).sum(1).mean()
print("images",tuple(demo_images.shape),"labels",tuple(demo_labels.shape),
      "soft targets",tuple(soft_targets.shape),"logits",tuple(demo_logits.shape),"loss",tuple(demo_loss.shape))
print("Ví dụ loss với logits ngẫu nhiên:",round(demo_loss.item(),4),"— chỉ minh họa công thức")
lr_cfg={**cfg,"epochs":600}
schedule=[learning_rate(lr_cfg,epoch,0,100) for epoch in range(600)]
fig,ax=plt.subplots(figsize=(8,2.7)); ax.plot(schedule)
ax.set(xlabel="Epoch",ylabel="LR",title="Warmup + cosine — lịch full 600 epoch")
plt.show(); plt.close(fig)

## 8. Train, validation, checkpoint và resume

`train_epoch`: zero_grad → forward → loss → backward → unscale/clip → optimizer step → EMA. Validation chạy `model.eval()` và `torch.no_grad()`; so sánh raw/EMA để chọn best theo validation.

Mỗi epoch hoàn tất train **và** validation mới ghi best (nếu cải thiện), last, metrics và một dòng log. Last chứa model/EMA/optimizer/scaler/RNG/history; không lưu batch dở hoặc checkpoint đánh số. Ghi bằng file tạm và atomic replace; best ghi trước last. Resume kiểm tra source cell thực thi/config/runtime/split. Kernel interrupt giữa epoch giữ last của epoch hoàn chỉnh trước đó.

In [ ]:
_remember_cell("engine-helpers")

@contextmanager
def run_lock(directory):
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / '.train.lock').open('a+b') as stream:
        if stream.tell() == 0:
            stream.write(b'0')
            stream.flush()
        stream.seek(0)
        try:
            if os.name == 'nt':
                import msvcrt
                msvcrt.locking(stream.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError as error:
            raise RuntimeError(f'Another trainer owns {directory}') from error
        try:
            yield
        finally:
            stream.seek(0)
            if os.name == 'nt':
                msvcrt.locking(stream.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(stream.fileno(), fcntl.LOCK_UN)

def atomic_write(path, writer, binary=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    try:
        with temporary.open('wb' if binary else 'w',
                            **({} if binary else {'encoding': 'utf-8', 'newline': ''})) as stream:
            writer(stream)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()

def atomic_json(path, payload):
    atomic_write(path, lambda stream: json.dump(payload, stream, indent=2, allow_nan=False))

def atomic_torch(path, payload):
    atomic_write(path, lambda stream: torch.save(payload, stream), binary=True)

def load_checkpoint(path):
    # Only deserialize checkpoints written by this trusted repository.
    return torch.load(path, map_location='cpu', weights_only=False)

def cpu_weights(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

def make_logger(directory):
    logger = logging.getLogger('cifar.' + str(directory))
    logger.setLevel(logging.INFO)
    logger.propagate = False
    formatter = logging.Formatter('%(asctime)s | %(message)s', datefmt='%H:%M:%S')
    for handler in (logging.StreamHandler(sys.stdout),
                    logging.FileHandler(directory / 'training.log', encoding='utf-8')):
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger

def close_logger(logger):
    for handler in list(logger.handlers):
        handler.close()
        logger.removeHandler(handler)

def duration(seconds):
    seconds = int(seconds)
    return f'{seconds//3600:02d}:{seconds//60%60:02d}:{seconds%60:02d}'

def train_epoch(model, ema, optimizer, scaler, source, split, config, epoch, global_step):
    loader = make_loader(source, split['training'], config, epoch, training=True)
    model.train()
    loss_sum = correct = seen = skips = 0
    for batch, (images, labels) in enumerate(loader):
        images, labels = images.to(config['device'], non_blocking=True), labels.to(config['device'], non_blocking=True)
        images, targets, paired, coefficient = mix_batch(images, labels, split['classes'], config, epoch)
        lr = learning_rate(config, epoch, batch, len(loader))
        for group in optimizer.param_groups:
            group['lr'] = lr
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(config['device'], enabled=scaler.is_enabled()):
            logits = model(images)
            loss = -(targets * F.log_softmax(logits.float(), dim=1)).sum(1).mean()
        if not torch.isfinite(loss):
            raise FloatingPointError('Non-finite training loss; last completed checkpoint preserved')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        skipped = scaler.get_scale() < previous_scale
        skips += int(skipped)
        if not skipped:
            global_step += 1
            update_ema(ema, model, global_step, config['ema_decay'])
        n = labels.numel()
        loss_sum += loss.item() * n
        prediction = logits.argmax(1)
        correct += (coefficient * prediction.eq(labels).sum().item()
                    + (1-coefficient) * prediction.eq(paired).sum().item())
        seen += n
    return {'train_loss': loss_sum/seen, 'train_mixed_accuracy': 100*correct/seen,
            'lr': lr, 'amp_skips': skips}, global_step

@torch.no_grad()
def evaluate(model, loader, device, extended=False):
    model.eval()
    loss_sum = correct = seen = 0
    confusion = None
    top5 = 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        logits = model(images)
        loss_sum += F.cross_entropy(logits, labels, reduction='sum').item()
        predictions = logits.argmax(1)
        correct += predictions.eq(labels).sum().item()
        seen += labels.numel()
        if extended:
            classes = logits.shape[1]
            if confusion is None:
                confusion = torch.zeros(classes, classes, dtype=torch.long)
            confusion += torch.bincount((labels*classes+predictions).cpu(), minlength=classes**2).reshape(classes, classes)
            top5 += logits.topk(min(5, classes), 1).indices.eq(labels[:, None]).any(1).sum().item()
    if seen == 0:
        raise ValueError('Evaluation split is empty')
    result = {'loss': loss_sum/seen, 'accuracy': 100*correct/seen}
    if extended:
        tp = confusion.diag().float()
        recall = tp / confusion.sum(1).clamp_min(1)
        precision = tp / confusion.sum(0).clamp_min(1)
        f1 = 2*precision*recall / (precision+recall).clamp_min(1e-12)
        result.update(top5_accuracy=100*top5/seen, macro_f1=f1.mean().item(),
                      per_class_recall=recall.tolist(), confusion_matrix=confusion.tolist())
    return result

def write_history(directory, history):
    if not history:
        return
    def write(stream):
        writer = csv.DictWriter(stream, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)
    atomic_write(directory / 'metrics.csv', write)

def validate_config(config):
    if config['phase'] not in ('selection', 'final'):
        raise ValueError('phase must be selection or final')
    if min(config['epochs'], config['batch_size']) < 1 or config['workers'] < 0:
        raise ValueError('Invalid epoch/batch/worker settings')
    if not 0 <= config['label_smoothing'] < 1 or not 0 <= config['mix_probability'] <= 1:
        raise ValueError('Invalid label smoothing or mixing probability')
    if config['lr'] <= 0 or config['weight_decay'] < 0 or config['grad_clip'] <= 0:
        raise ValueError('Invalid learning rate, weight decay or gradient clipping')
    if config['device'] == 'auto':
        config['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
    if config['device'] == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable in the selected Python; use --device cpu for debugging')

In [ ]:
_remember_cell("notebook-fingerprint")

def source_fingerprint():
    # Không hash outputs/execution_count; re-run không làm mất khả năng resume.
    # Hash source các cell định nghĩa đã chạy; sửa chưa Save cũng được nhận biết.
    required = {"bootstrap","data-definitions","shape-helpers","whole-model",
                "profile-definitions","recipe-definitions","engine-helpers",
                "notebook-fingerprint","training-engine","final-definitions"}
    required.update(f"block{i:02}" for i in range(1,14))
    if required - NOTEBOOK_TRAINING_CELLS.keys():
        raise RuntimeError("Chưa chạy đủ cell định nghĩa trước train: " + str(sorted(required-NOTEBOOK_TRAINING_CELLS.keys())))
    raw = json.dumps(NOTEBOOK_TRAINING_CELLS, sort_keys=True,ensure_ascii=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def environment(config):
    return {"python":platform.python_version(),"pytorch":torch.__version__,
            "torchvision":torchvision.__version__,"numpy":np.__version__,
            "cuda":torch.version.cuda,"device":config["device"],"platform":platform.system(),
            "gpu":torch.cuda.get_device_name(0) if config["device"]=="cuda" else None}

In [ ]:
_remember_cell("training-engine")

def run_experiment(config, stop_after_epochs=None, epoch_callback=None):
    config = copy.deepcopy(config)
    validate_config(config)
    directory = Path(config['run_dir']).resolve()
    config['run_dir'] = str(directory)
    with run_lock(directory):
        logger = make_logger(directory)
        writer = None
        try:
            identity, env = source_fingerprint(), environment(config)
            last = directory / 'last.pt'
            if not last.exists() and any((directory / name).exists()
                                         for name in ('best.pt', 'metrics.csv', 'result.json')):
                raise ValueError('Run artifacts exist but last.pt is missing; existing files preserved. '
                                 'Restore last.pt or use a new --run-name')
            previous = load_checkpoint(last) if last.exists() else None
            if previous:
                if previous['version'] != CHECKPOINT_VERSION or previous['source_sha256'] != identity:
                    raise ValueError('Changed training source/version; use a new --run-name, existing checkpoints preserved')
                if previous['config'] != config:
                    raise ValueError('Changed configuration; resume rejected. Restore settings or use --run-name')
                if previous['environment'] != env:
                    raise ValueError('Changed runtime environment; exact resume requires the recorded environment')
                if (directory / 'result.json').exists() and previous['epoch'] == config['epochs']:
                    return json.loads((directory / 'result.json').read_text(encoding='utf-8'))
            seed_everything(config['seed'], config['deterministic'])
            source, split = prepare_data(config)
            if previous and previous['split_sha256'] != split['sha256']:
                raise ValueError('Changed dataset split; resume rejected')
            model = make_model(split['classes'])
            profile = inspect_model(model, split['classes'], backward=False)
            model = model.to(config['device'])
            ema = copy.deepcopy(model).eval().requires_grad_(False)
            optimizer = make_optimizer(model, config)
            scaler = torch.amp.GradScaler('cuda', enabled=config['device'] == 'cuda' and config['amp'], init_scale=4096.)
            history, best_weights = [], None
            best_needs_repair = False
            epoch, global_step, best_accuracy, best_epoch, best_kind, elapsed_seconds = 0, 0, -1., 0, 'raw', 0.
            if previous:
                model.load_state_dict(previous['model'])
                ema.load_state_dict(previous['ema'])
                optimizer.load_state_dict(previous['optimizer'])
                scaler.load_state_dict(previous['scaler'])
                epoch, global_step = previous['epoch'], previous['global_step']
                history = previous['history']
                best_accuracy, best_epoch, best_kind = previous['best_accuracy'], previous['best_epoch'], previous['best_kind']
                best_weights, elapsed_seconds = previous['best_weights'], previous['elapsed_seconds']
                restore_rng(previous['rng'])
                best_path = directory / 'best.pt'
                try:
                    best_document = load_checkpoint(best_path) if best_path.exists() else None
                except Exception:
                    best_document = None
                if (best_document is None or best_document['epoch'] != best_epoch
                        or best_document['accuracy'] != best_accuracy):
                    best_needs_repair = True
            try:
                git_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True, stderr=subprocess.DEVNULL).strip()
            except (OSError, subprocess.CalledProcessError):
                git_commit = 'unavailable'
            atomic_json(directory / 'config.json', config)
            atomic_json(directory / 'split.json', split)
            atomic_json(directory / 'metadata.json', {'environment': env, 'python_executable': sys.executable,
                        'git_commit': git_commit, 'source_sha256': identity, 'profile': profile,
                        'started_at_utc': previous['started_at_utc'] if previous else datetime.now(timezone.utc).isoformat()})
            metadata = json.loads((directory / 'metadata.json').read_text(encoding='utf-8'))
            completed_this_call = 0
            while epoch < config['epochs']:
                started = time.monotonic()
                if config['device'] == 'cuda':
                    torch.cuda.reset_peak_memory_stats()
                metrics, global_step = train_epoch(model, ema, optimizer, scaler, source, split, config, epoch, global_step)
                if config['phase'] == 'selection':
                    loader = make_loader(source, split['validation'], config, 0)
                    raw = evaluate(model, loader, config['device'])
                    averaged = evaluate(ema, loader, config['device'])
                    score, kind, selected = raw['accuracy'], 'raw', model
                    if averaged['accuracy'] > score:
                        score, kind, selected = averaged['accuracy'], 'ema', ema
                    metrics.update(val_loss=raw['loss'], val_accuracy=raw['accuracy'],
                                   ema_val_loss=averaged['loss'], ema_val_accuracy=averaged['accuracy'])
                    improved = score > best_accuracy
                else:
                    selected = ema if config['final_weights'] == 'ema' else model
                    score, kind, improved = -1., config['final_weights'] + '_fixed_epoch', True
                epoch += 1
                if improved:
                    best_accuracy, best_epoch, best_kind, best_weights = score, epoch, kind, cpu_weights(selected)
                record = {'epoch': epoch, **metrics, 'best_val_accuracy': best_accuracy, 'best_epoch': best_epoch}
                history.append(record)
                seconds = time.monotonic() - started
                elapsed_seconds += seconds
                if config['tensorboard'] and writer is None:
                    from torch.utils.tensorboard import SummaryWriter
                    writer = SummaryWriter(str(directory / 'tensorboard'), purge_step=epoch)
                payload = {'version': CHECKPOINT_VERSION, 'source_sha256': identity, 'environment': env,
                           'config': config, 'split_sha256': split['sha256'], 'epoch': epoch,
                           'global_step': global_step, 'model': cpu_weights(model), 'ema': cpu_weights(ema),
                           'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                           'rng': capture_rng(), 'history': history, 'best_accuracy': best_accuracy,
                           'best_epoch': best_epoch, 'best_kind': best_kind, 'best_weights': best_weights,
                           'elapsed_seconds': elapsed_seconds, 'started_at_utc': metadata['started_at_utc']}
                best_saved = improved or best_needs_repair
                if best_saved:
                    atomic_torch(directory / 'best.pt', {'model': best_weights, 'epoch': best_epoch,
                                 'accuracy': best_accuracy, 'kind': best_kind, 'config': config})
                    best_needs_repair = False
                atomic_torch(last, payload)
                write_history(directory, history)
                if writer:
                    for key, value in record.items():
                        writer.add_scalar(key, value, epoch)
                    writer.flush()
                vram = torch.cuda.max_memory_allocated()/2**30 if config['device'] == 'cuda' else 0.
                logger.info('%s %s %s/%s(base_lr=%g) | EPOCH %d/%d | loss=%.4f mixed_acc=%.2f%% val_raw=%s val_ema=%s best=%s @%d | lr=%.6g time=%s total_ETA=%s VRAM=%.2fGB AMP_skips=%d | saved last.pt%s',
                            config['dataset'], config['phase'], config['model']['family'], config['optimizer'], config['lr'],
                            epoch, config['epochs'], metrics['train_loss'], metrics['train_mixed_accuracy'],
                            f"{metrics['val_accuracy']:.2f}%" if 'val_accuracy' in metrics else 'sealed',
                            f"{metrics['ema_val_accuracy']:.2f}%" if 'ema_val_accuracy' in metrics else 'sealed',
                            f'{best_accuracy:.2f}%' if best_accuracy >= 0 else 'fixed final epoch', best_epoch,
                            metrics['lr'], duration(seconds), duration(elapsed_seconds/epoch*(config['epochs']-epoch)),
                            vram, metrics['amp_skips'], ' + best.pt' if best_saved else '')
                completed_this_call += 1
                if epoch_callback:
                    epoch_callback(epoch, best_accuracy)
                if REQUEST_STOP or (stop_after_epochs and completed_this_call >= stop_after_epochs):
                    if epoch < config['epochs']:
                        return {'status': 'interrupted', 'epoch': epoch, 'history': history}
                    break
            result = {'status': 'complete', 'experiment_id': directory.name, 'dataset': config['dataset'],
                      'model_name': config['model']['family'], 'architecture_version': 'compactmix-notebook-v1',
                      'seed': config['seed'], 'parameters': profile['parameters'], 'flops': profile['flops'],
                      'optimizer': config['optimizer'], 'learning_rate': config['lr'], 'weight_decay': config['weight_decay'],
                      'scheduler': config['scheduler'], 'augmentation': config['augmentation'], 'epochs': epoch,
                      'best_val_accuracy': best_accuracy if config['phase'] == 'selection' else None,
                      'best_epoch': best_epoch, 'best_kind': best_kind,
                      'final_val_accuracy': max(history[-1]['val_accuracy'], history[-1]['ema_val_accuracy']) if config['phase'] == 'selection' else None,
                      'test_accuracy': None, 'training_time': elapsed_seconds, 'checkpoint': str(directory / 'best.pt'),
                      'synthetic': split['synthetic'], 'subset': split['subset'], 'phase': config['phase'],
                      'history': history, 'config': config, 'profile': profile, 'metadata': metadata,
                      'notes': 'Synthetic/subset checks are not benchmark evidence' if split['synthetic'] or split['subset'] else ''}
            atomic_json(directory / 'result.json', result)
            return result
        finally:
            if writer:
                writer.close()
            close_logger(logger)

In [ ]:
_remember_cell("final-definitions")

def read_result(directory):
    path = Path(directory)/"result.json"
    if not path.exists():
        raise ValueError("Run chưa hoàn tất, chưa có result.json.")
    return json.loads(path.read_text(encoding="utf-8"))

def evaluate_final(directory, verbose=True):
    directory = Path(directory).resolve()
    with run_lock(directory):
        result = read_result(directory)
        config = result["config"]
        freeze = json.loads((directory/"frozen.json").read_text(encoding="utf-8"))
        if result["status"]!="complete" or result["phase"]!="final":
            raise ValueError("Test yêu cầu completed frozen final run.")
        if result["synthetic"] or result["subset"]:
            raise ValueError("Smoke/subset không được dùng làm benchmark test.")
        if freeze["source_sha256"]!=source_fingerprint() or freeze["config"]!=config:
            raise ValueError("Source/config đã đổi sau freeze; không đánh giá checkpoint khác kiến trúc.")
        if result["metadata"]["environment"] != environment(config):
            raise ValueError("Runtime đã đổi so với final run.")
        checkpoint_path=directory/"best.pt"
        digest=hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
        if result["test_accuracy"] is not None:
            if result["test_checkpoint_sha256"]!=digest:
                raise ValueError("Checkpoint đã đổi sau test; không tái sử dụng điểm số.")
        else:
            classes=10 if config["dataset"]=="cifar10" else 100
            model=make_model(classes).to(config["device"])
            model.load_state_dict(load_checkpoint(checkpoint_path)["model"],strict=True)
            constructor=datasets.CIFAR10 if classes==10 else datasets.CIFAR100
            source=constructor(root=config["data_root"],train=False,download=config["download"])
            loader=make_loader(source,list(range(len(source))),config,0,training=False)
            metrics=evaluate(model,loader,config["device"],extended=True)
            result.update(test_accuracy=metrics["accuracy"],test_metrics=metrics,
                          test_checkpoint_sha256=digest)
            atomic_json(directory/"result.json",result)
        if verbose:
            print(json.dumps({"dataset":config["dataset"],"test_accuracy":result["test_accuracy"],
                              "checkpoint_sha256":digest},ensure_ascii=False))
        return result

def finalize(selection_directory, stop_after_epochs=None):
    selection_directory=Path(selection_directory).resolve()
    result=read_result(selection_directory)
    if result["status"]!="complete" or result["phase"]!="selection":
        raise ValueError("Cần selection hoàn tất để freeze và retrain final.")
    if result["synthetic"] or result["subset"]:
        raise ValueError("Không finalize smoke/subset thành benchmark.")
    if result["metadata"]["source_sha256"] != source_fingerprint():
        raise ValueError("Selection source đã đổi; chạy selection mới với --run-name mới trước khi freeze.")
    if result["metadata"]["environment"] != environment(result["config"]):
        raise ValueError("Selection runtime đã đổi; không freeze recipe của runtime khác.")
    config=copy.deepcopy(result["config"])
    config.update(phase="final",epochs=result["best_epoch"],final_weights=result["best_kind"],
                  run_dir=str(selection_directory.with_name(selection_directory.name+"_final")))
    directory=Path(config["run_dir"])
    freeze={"selection_run":str(selection_directory),"selection_best_epoch":result["best_epoch"],
            "selection_validation_accuracy":result["best_val_accuracy"],"config":config,
            "source_sha256":source_fingerprint()}
    path=directory/"frozen.json"
    directory.mkdir(parents=True,exist_ok=True)
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8"))!=freeze:
            raise ValueError("Final recipe/source đã đổi; giữ checkpoint cũ, dùng --run-name mới.")
    else:
        atomic_json(path,freeze)
    trained=run_experiment(config,stop_after_epochs)
    if trained["status"]=="interrupted" or REQUEST_STOP:
        return trained
    return evaluate_final(directory,verbose=False)

### Chạy train mới / chạy tiếp

**Cell này là điểm bắt đầu train.** Ở demo, mỗi lần chạy thêm tối đa một epoch cho mỗi dataset. Chạy lại đúng cell để tiếp tục epoch 2; không đổi EPOCHS hoặc source để minh họa resume. Chạy lại khi đủ 2 epoch không train thêm.

Mặc định dùng ảnh CIFAR thật nhưng là subset nhỏ. Giá trị demo không thay thế kết quả full-data trong PDF. `MODE="full"` sẽ chạy selection 600 epoch; `MODE="report_only"` bỏ qua cell train. Có thể nhấn Interrupt Kernel để dừng; lần sau phát lại epoch chưa hoàn tất từ last.pt.

In [ ]:
def run_requested_training():
    global REQUEST_STOP
    REQUEST_STOP=False
    results={}
    if MODE=="report_only":
        return results
    for cfg in notebook_configs():
        # Preflight im lặng, không sửa shape và không dùng model kiểm tra để train.
        inspect_model(make_model(10 if cfg["dataset"]=="cifar10" else 100),
                      10 if cfg["dataset"]=="cifar10" else 100,backward=True)
        results[cfg["dataset"]]=run_experiment(cfg,
            stop_after_epochs=DEMO_STOP_AFTER if MODE=="demo" else None)
    return results

NEW_RESULTS=run_requested_training()

### Kiểm tra last/best và vẽ lịch sử của lượt vừa chạy

Bảng này chỉ đọc checkpoint sau epoch hoàn chỉnh. Best của selection dùng validation; log train không in theo batch.

In [ ]:
def plot_history(history,title,filename):
    frame=pd.DataFrame(history)
    if frame.empty:
        return
    fig,axes=plt.subplots(1,3,figsize=(14,3.3))
    for key,label in (("train_loss","Train soft-label"),("val_loss","Val raw"),("ema_val_loss","Val EMA")):
        if key in frame: axes[0].plot(frame.epoch,frame[key],label=label)
    for key,label in (("train_mixed_accuracy","Train mixed"),("val_accuracy","Val raw"),("ema_val_accuracy","Val EMA")):
        if key in frame: axes[1].plot(frame.epoch,frame[key],label=label)
    if "lr" in frame: axes[2].plot(frame.epoch,frame.lr)
    for ax in axes:
        ax.set_xlabel("Epoch"); ax.grid(alpha=.2)
    axes[0].set_ylabel("Loss"); axes[1].set_ylabel("Accuracy (%)"); axes[2].set_ylabel("LR")
    axes[0].legend(); axes[1].legend(); fig.suptitle(title)
    fig.tight_layout(); fig.savefig(OUTPUT_ROOT/filename,bbox_inches="tight")
    plt.show(); plt.close(fig)

checkpoint_rows=[]
for cfg in notebook_configs():
    folder=Path(cfg["run_dir"])
    if not (folder/"last.pt").exists(): continue
    last=load_checkpoint(folder/"last.pt"); best=load_checkpoint(folder/"best.pt")
    assert {p.name for p in folder.glob("*.pt")}=={"best.pt","last.pt"}
    checkpoint_rows.append({"dataset":cfg["dataset"],"last epoch":last["epoch"],
        "best epoch":best["epoch"],"best val %":best["accuracy"],"weights":best["kind"],
        "run":folder.name,"subset":bool(cfg["limit_train"] or cfg["limit_val"]),"synthetic":cfg["smoke"]})
    plot_history(last["history"],cfg["dataset"]+" — lượt mới, "+MODE,cfg["dataset"]+"_new_curves.png")
if checkpoint_rows: display(pd.DataFrame(checkpoint_rows))

## 9. Freeze recipe, retrain 50k, official test

Mặc định **không chạy** phần dài này (`RUN_FINAL=False`). Sau selection full hoàn tất, freeze best epoch và loại weights raw/EMA, khởi tạo lại và train trên toàn bộ 50k. Official test chỉ được mở sau khi final train hoàn tất; SHA checkpoint gắn với test result và điểm số được cache. Final best là checkpoint của epoch cố định, không chọn bằng validation/test.

Demo/synthetic/subset bị từ chối finalize. Code final có đầy đủ phía trên để kiểm tra; không gán test score của best 45k cho final 50k.

In [ ]:
FINAL_RESULTS = {}
if RUN_FINAL:
    if MODE!="full" or SYNTHETIC_CHECK:
        raise ValueError("Final cần MODE=full và dữ liệu đầy đủ, không dùng demo/synthetic")
    for cfg in notebook_configs("full"):
        FINAL_RESULTS[cfg["dataset"]] = finalize(cfg["run_dir"])
    display(pd.DataFrame([{"dataset":name,"phase":result["phase"],
                           "epochs":result["epochs"],"test_accuracy":result["test_accuracy"]}
                          for name,result in FINAL_RESULTS.items()]))

## 10. Kết quả đã đo trong PDF: kiểm chứng từ evidence

Các cell tiếp theo đọc `report/evidence/`, `report/checkpoints/` và `results_summary.json`. Chúng kiểm tra SHA256 best, 600 dòng lịch sử liên tục, split 45k/5k disjoint, số ảnh test và số đúng trong confusion matrix. Đây là **đọc bằng chứng cũ**, không mở test loader mới.

Nếu chỉ sao chép notebook sang thư mục khác, phần model/train vẫn có đủ code; muốn trình bày kết quả cũ thì mang kèm thư mục report/evidence, report/checkpoints và results_summary.json. Dataset đặt trong datasets/.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def archived_bundle(name):
    folder=REPORT_DIR/"evidence"/name
    required=[folder/"metrics.csv",folder/"split.json",folder/"test_evaluation.json",
              folder/"result.json",folder/"config.json",REPORT_DIR/"checkpoints"/name/"best.pt"]
    missing=[str(p) for p in required if not p.exists()]
    if missing: raise FileNotFoundError("Thiếu evidence cũ: "+", ".join(missing))
    history=pd.read_csv(folder/"metrics.csv")
    split=read_json(folder/"split.json")
    test=read_json(folder/"test_evaluation.json")
    result=read_json(folder/"result.json")
    checkpoint=REPORT_DIR/"checkpoints"/name/"best.pt"
    digest=hashlib.sha256(checkpoint.read_bytes()).hexdigest()
    if digest!=test["checkpoint_sha256"]: raise ValueError("Archived best SHA mismatch: "+name)
    assert history.epoch.tolist()==list(range(1,601))
    assert len(split["training"])==45000 and len(split["validation"])==5000
    assert not set(split["training"]) & set(split["validation"])
    assert sorted(split["training"]+split["validation"])==list(range(50000))
    cm=np.asarray(test["test_metrics"]["confusion_matrix"])
    assert int(cm.sum())==test["samples"]==10000
    assert int(cm.trace())==test["correct"]
    assert math.isclose(100*cm.trace()/cm.sum(),test["test_metrics"]["accuracy"],abs_tol=1e-9)
    assert test["training_images"]==45000 and test["is_frozen_final_retraining"] is False
    return {"history":history,"split":split,"test":test,"result":result,
            "config":read_json(folder/"config.json"),"checkpoint":checkpoint,
            "per_class":pd.read_csv(folder/"per_class_metrics.csv")}

ARCHIVED={}
archive_rows=[]
for name in DATASETS:
    bundle=archived_bundle(name); ARCHIVED[name]=bundle
    result,test=bundle["result"],bundle["test"]
    archive_rows.append({"dataset":name,"scope":"Archived best train 45k",
        "best epoch":test["epoch"],"weights":test["weight_kind"],
        "validation top1 %":test["validation_accuracy"],"test top1 %":test["test_metrics"]["accuracy"],
        "test top5 %":test["test_metrics"]["top5_accuracy"],"test macro-F1":test["test_metrics"]["macro_f1"],
        "correct/10000":test["correct"],"incorrect":test["incorrect"],
        "parameters":result["parameters"],"FLOPs":result["flops"],"SHA verified":True})
ARCHIVED_TABLE=pd.DataFrame(archive_rows)
display(ARCHIVED_TABLE)
ARCHIVED_TABLE.to_csv(OUTPUT_ROOT/"archived_summary.csv",index=False)

### Đường học của 600 epoch và cấu hình train đã dùng

Train loss có nhãn mềm; validation loss có nhãn cứng. Đường best được chọn bằng validation. Thời gian là tổng engine đã lưu, không cộng khoảng nghỉ giữa các buổi train.

In [ ]:
for name,bundle in ARCHIVED.items():
    test,result=bundle["test"],bundle["result"]
    plot_history(bundle["history"],name+f" — archive 600 epoch, best @{test['epoch']}",name+"_archived_curves.png")
    config=pd.DataFrame({"setting":list(bundle["config"]),"value":[str(v) for v in bundle["config"].values()]})
    display(Markdown("**"+name+" — learning settings đã lưu**")); display(config)
    display(pd.DataFrame([{"epoch count":result["epochs"],"best epoch":result["best_epoch"],
        "training seconds":result["training_time"],"mean seconds/epoch":result["training_time"]/result["epochs"],
        "AMP skips":int(bundle["history"].amp_skips.sum()),"test seconds":test["evaluation_seconds"],
        "test precision":test["precision"],"test batch":test["evaluation_batch_size"],
        "test augmentation":test["augmentation"],"TTA":test["test_time_augmentation"]}]))

### Test top-1/top-5, confusion matrix và precision/recall/F1

`accuracy=100*correct/total`; `precision_c=TP_c/(TP_c+FP_c)`; `recall_c=TP_c/(TP_c+FN_c)`; `F1_c=2PR/(P+R)`; macro-F1 lấy trung bình các lớp. Hàng confusion là lớp thật, cột là lớp dự đoán. Top-5 kiểm tra nhãn thật nằm trong 5 logits cao nhất.

C10 vẽ đủ số đếm; C100 vẽ đủ ma trận 100×100, thêm 15 lớp recall thấp nhất để dễ đọc. Bảng đầy đủ tất cả lớp được xuất CSV.

In [ ]:
def per_class_from_confusion(cm,names):
    cm=np.asarray(cm,dtype=np.int64)
    tp=np.diag(cm); support=cm.sum(1); predicted=cm.sum(0)
    recall=tp/np.maximum(support,1); precision=tp/np.maximum(predicted,1)
    f1=2*precision*recall/np.maximum(precision+recall,1e-12)
    return pd.DataFrame({"class_id":range(len(cm)),"class_name":names,"support":support,
        "correct":tp,"precision_percent":100*precision,"recall_percent":100*recall,"f1_percent":100*f1})

for name,bundle in ARCHIVED.items():
    cm=np.asarray(bundle["test"]["test_metrics"]["confusion_matrix"])
    names=bundle["per_class"].class_name.tolist()
    per_class=per_class_from_confusion(cm,names)
    assert math.isclose(per_class.f1_percent.mean()/100,bundle["test"]["test_metrics"]["macro_f1"],abs_tol=1e-6)
    np.testing.assert_allclose(per_class.recall_percent,bundle["per_class"].recall_percent,atol=1e-6)
    per_class.to_csv(OUTPUT_ROOT/f"{name}_per_class.csv",index=False,encoding="utf-8-sig")
    pd.DataFrame(cm,index=names,columns=names).to_csv(OUTPUT_ROOT/f"{name}_confusion.csv",encoding="utf-8-sig")
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    im=axes[0].imshow(100*cm/np.maximum(cm.sum(1,keepdims=True),1),vmin=0,vmax=100,cmap="Blues")
    fig.colorbar(im,ax=axes[0],label="% theo hàng")
    axes[0].set(xlabel="Predicted class",ylabel="True class",title=name+" — đủ "+str(len(cm))+" lớp")
    if len(cm)==10:
        axes[0].set_xticks(range(10),names,rotation=70); axes[0].set_yticks(range(10),names)
        for i in range(10):
            for j in range(10):
                axes[0].text(j,i,str(cm[i,j]),ha="center",va="center",fontsize=6,color="white" if i==j else "black")
    worst=per_class.nsmallest(min(15,len(cm)),"recall_percent").sort_values("recall_percent")
    axes[1].barh(worst.class_name,worst.recall_percent,color="#397bb5")
    axes[1].set(xlim=(0,100),xlabel="Recall (%)",title="Các lớp recall thấp nhất")
    fig.tight_layout(); fig.savefig(OUTPUT_ROOT/f"{name}_archived_confusion_recall.png",bbox_inches="tight")
    plt.show(); plt.close(fig)
    display(Markdown("**"+name+" — toàn bộ chỉ số từng lớp**"))
    with pd.option_context("display.max_rows",110): display(per_class)

## 11. Nạp best 45k và dự đoán ảnh để minh họa

Trọng số cũ có tên layer từ `src/models.py`, còn notebook dùng tên tường minh. Hàm dưới ánh xạ **tên tương ứng của đúng kiến trúc** và nạp `strict=True`; không nạp bằng thứ tự tùy ý hoặc bỏ qua weight không khớp. Nếu tự đổi kiến trúc, không dùng checkpoint cũ để gán điểm số cho model mới.

Cell minh họa dùng **ảnh validation từ official training data**, không mở test để lựa chọn kiến trúc. Đây là ví dụ dự đoán, không phải một phép đo accuracy toàn tập mới.

In [ ]:
def load_archived_model(name):
    bundle=ARCHIVED[name]
    checkpoint=load_checkpoint(bundle["checkpoint"])
    model=make_model(10 if name=="cifar10" else 100).eval()
    profile=inspect_model(model,10 if name=="cifar10" else 100,backward=False)
    assert profile["parameters"]==bundle["result"]["parameters"]
    assert profile["flops"]==bundle["result"]["flops"]
    prefixes={"stem.0":"stem","stages.1.0":"t1_bn","stages.1.3":"t1_conv",
              "stages.2.0":"t2_bn","stages.2.3":"t2_conv","head.0":"head_bn","head.4":"classifier"}
    suffixes={"pre.0":"bn_pre","pre.2":"expand","pre.3":"bn_expand",
              "local":"local","context":"context","mix.0":"bn_mix","mix.2":"grouped",
              "mix.3":"bn_gate","mix.5.gate.0":"se_reduce","mix.5.gate.2":"se_expand","mix.6":"project"}
    block_locations=[(0,0),(0,1),(1,4),(1,5),(1,6),*[(2,i) for i in range(4,12)]]
    for number,(stage,position) in enumerate(block_locations,1):
        for old,new in suffixes.items():
            prefixes[f"stages.{stage}.{position}.{old}"]=f"block{number:02}.{new}"
    converted={}
    for old_key,value in checkpoint["model"].items():
        prefix,suffix=old_key.rsplit(".",1)
        if prefix not in prefixes: raise ValueError("Không có ánh xạ weight: "+old_key)
        new_key=prefixes[prefix]+"."+suffix
        if new_key in converted: raise ValueError("Weight mapping trùng: "+new_key)
        converted[new_key]=value
    model.load_state_dict(converted,strict=True)
    return model.to(DEVICE).eval()

In [ ]:
for name,bundle in ARCHIVED.items():
    model=load_archived_model(name)
    cfg={**bundle["config"],"workers":0,"batch_size":8,"device":DEVICE,
         "data_root":str(DATA_ROOT),"download":False,"smoke":False,"limit_train":0,"limit_val":0}
    constructor=datasets.CIFAR10 if name=="cifar10" else datasets.CIFAR100
    source=constructor(root=str(DATA_ROOT),train=True,download=False)
    indices=bundle["split"]["validation"][:8]
    images,labels=next(iter(make_loader(source,indices,cfg,0,training=False)))
    with torch.no_grad(): predictions=model(images.to(DEVICE)).argmax(1).cpu()
    fig,axes=plt.subplots(2,4,figsize=(11,5))
    rows=[]
    for ax,index,truth,prediction in zip(axes.flat,indices,labels.tolist(),predictions.tolist()):
        image,_=source[index]; ax.imshow(image); ax.axis("off")
        ax.set_title("True: "+source.classes[truth]+"\nPred: "+source.classes[prediction],
                     color="green" if truth==prediction else "red",fontsize=9)
        rows.append({"index training set":index,"true":source.classes[truth],"predicted":source.classes[prediction]})
    fig.suptitle(name+" — archived best, 8 ảnh VALIDATION để minh họa")
    fig.tight_layout(); fig.savefig(OUTPUT_ROOT/f"{name}_validation_predictions.png",bbox_inches="tight")
    plt.show(); plt.close(fig); display(pd.DataFrame(rows)); del model

### Tùy chọn: đo lại best cũ trên đủ 10k test

`RECHECK_ARCHIVED_TEST=False` mặc định để dùng kết quả test đã có trong PDF. Nếu chủ động bật True, đánh giá lại đúng checkpoint đã mở test trước đây, với hash/kiến trúc được kiểm tra. Không dùng điểm số này để chỉnh kiến trúc. Đây vẫn là test của **best train 45k**, không phải frozen final 50k. Kết quả mới được lưu riêng, không ghi đè evidence cũ.

In [ ]:
LIVE_ARCHIVED_TEST={}
if RECHECK_ARCHIVED_TEST:
    for name,bundle in ARCHIVED.items():
        digest=hashlib.sha256(bundle["checkpoint"].read_bytes()).hexdigest()
        if digest!=bundle["test"]["checkpoint_sha256"]: raise ValueError("Archived checkpoint đã đổi")
        cache=OUTPUT_ROOT/f"{name}_archived_live_test.json"
        if cache.exists():
            measured=read_json(cache)
            if measured["checkpoint_sha256"]!=digest or measured["source_sha256"]!=source_fingerprint():
                raise ValueError("Cache test không khớp source/checkpoint; giữ file để kiểm tra")
        else:
            model=load_archived_model(name)
            constructor=datasets.CIFAR10 if name=="cifar10" else datasets.CIFAR100
            source=constructor(root=str(DATA_ROOT),train=False,download=False)
            assert len(source)==10000
            cfg={**bundle["config"],"workers":0,"batch_size":64,"device":DEVICE}
            started=time.monotonic()
            metrics=evaluate(model,make_loader(source,list(range(len(source))),cfg,0),DEVICE,extended=True)
            measured={"scope":"archived selection45k test, not final50k","dataset":name,"samples":len(source),
                "checkpoint_sha256":digest,"source_sha256":source_fingerprint(),"metrics":metrics,
                "seconds":time.monotonic()-started,"environment":environment(cfg)}
            atomic_json(cache,measured); del model
        LIVE_ARCHIVED_TEST[name]=measured
    display(pd.DataFrame([{"dataset":n,"scope":r["scope"],"samples":r["samples"],
                          **{k:r["metrics"][k] for k in ("accuracy","top5_accuracy","macro_f1")}}
                         for n,r in LIVE_ARCHIVED_TEST.items()]))

## 12. So sánh optimizer/LR theo yêu cầu đề

PDF mới có full-data SGD LR=0,1 trên hai dataset. Các cấu hình còn lại dưới đây là **kế hoạch/chưa chạy**, không phải kết quả đã đạt. `RUN_MATRIX=False` mặc định; bật khi thật sự muốn chạy nhiều thí nghiệm. Mỗi hàng có run_dir riêng, giữ best/last và resume riêng. Matrix chỉ chọn bằng validation, không tự test từng candidate.

In [ ]:
MATRIX=[("sgd",.05),("sgd",.1),("sgd",.15),("adam",.0003),("adam",.001),("adam",.003)]
comparison=[]
for optimizer,lr in MATRIX:
    row={"optimizer":optimizer,"initial LR":lr}
    for name in DATASETS:
        row[name+" archived val/test %"]=(f"{ARCHIVED[name]['test']['validation_accuracy']:.2f} / "
            f"{ARCHIVED[name]['test']['test_metrics']['accuracy']:.2f}" if optimizer=="sgd" and lr==.1 else "Chưa có full-data trong PDF")
    comparison.append(row)
display(pd.DataFrame(comparison))
MATRIX_RESULTS=[]
if RUN_MATRIX:
    for optimizer,lr in MATRIX:
        for cfg in notebook_configs("full",optimizer,lr):
            result=run_experiment(cfg)
            MATRIX_RESULTS.append(result)
    display(pd.DataFrame([{"dataset":r["dataset"],"optimizer":r["optimizer"],"lr":r["learning_rate"],
                           "validation %":r["best_val_accuracy"],"test %":r["test_accuracy"]} for r in MATRIX_RESULTS]))

## 13. Đo tốc độ suy luận trên thiết bị hiện tại

Đo forward bằng tensor giả chỉ để đo thời gian, **không phải accuracy**. Batch=1, FP32, warmup 5 lần, đo 20 lần, đồng bộ CUDA quanh phép đo. Latency này không có trong PDF cũ và có thể thay đổi theo tải máy; không suy ra latency từ FLOPs.

In [ ]:
timing_rows=[]
for name in DATASETS:
    model=make_model(10 if name=="cifar10" else 100).to(DEVICE).eval()
    sample=torch.zeros(1,3,32,32,device=DEVICE)
    def sync():
        if DEVICE=="cuda": torch.cuda.synchronize()
    times=[]
    with torch.no_grad():
        for _ in range(5): model(sample)
        sync()
        for _ in range(20):
            sync(); start=time.perf_counter(); model(sample); sync()
            times.append(1000*(time.perf_counter()-start))
    timing_rows.append({"dataset":name,"device":DEVICE,"batch":1,"dtype":"FP32","repetitions":20,
        "median ms":float(np.median(times)),"p95 ms":float(np.percentile(times,95)),
        "mean images/s":1000/float(np.mean(times)),"scope":"technical forward timing, not accuracy"})
    del model
TIMING_TABLE=pd.DataFrame(timing_rows); display(TIMING_TABLE)
TIMING_TABLE.to_csv(OUTPUT_ROOT/"live_inference_timing.csv",index=False)

## 14. Các ý cần nói khi vấn đáp

| Câu hỏi | Ý trả lời |
|---|---|
| Vì sao hai kernel 3×3/5×5? | Hai phạm vi không gian trên hai phần kênh, concat lại trước grouped mixing. Đây là ý đồ thiết kế; chưa có ablation đo riêng lợi ích. |
| Conv groups=8 làm gì? | Chia kết nối theo nhóm, giảm weight/MAC so với dense Conv cùng kích thước. C_in,C_out phải chia hết cho 8. |
| SE có giảm kênh không? | Pool chỉ giảm H/W về 1; Conv reduce/expand mới đổi số kênh để tạo gate. |
| ADD khác concat? | ADD yêu cầu cùng shape và giữ C; concat theo kênh cộng C, đòi N/H/W giống nhau. |
| BN train khác eval? | Train dùng thống kê batch và cập nhật running stats; eval dùng thống kê đã lưu. |
| SiLU làm gì? | Phi tuyến x*sigmoid(x), không đổi shape hoặc có weight học riêng. |
| EMA và raw? | EMA là trung bình trọng số theo thời gian; ở selection so cả raw/EMA bằng validation để chọn best. |
| Last và best? | Last để resume đủ optimizer/scaler/RNG; best selection có validation accuracy cao nhất. Cả hai chỉ cập nhật sau epoch hoàn chỉnh. |
| Điểm trong PDF từ đâu? | Hai checkpoint best train 45k, test trên đủ 10k và kiểm chứng bằng SHA/confusion; không phải demo hoặc final 50k. |
| Đã đủ đề chưa? | Còn so sánh full-data Adam/nhiều LR, multi-seed/ablation và final-test 50k trong snapshot báo cáo. Không điền số liệu chưa chạy. |

Thư mục output của notebook chứa profile/layer CSV, hình kiến trúc, learning curves, confusion/per-class CSV, receipt và checkpoint lượt mới. PDF gốc là `report/Bao_cao_CIFAR10_CIFAR100.pdf`; không tự thay kết quả snapshot bằng accuracy demo.

In [ ]:
receipt={"created_at_utc":datetime.now(timezone.utc).isoformat(),"training_source_sha256":source_fingerprint(),
    "mode":MODE,"new_run_config":notebook_configs(),"runtime":environment(CONFIGS[0]),
    "profiles":PROFILE_TABLE.to_dict(orient="records"),"historical_archived_results":archive_rows,
    "live_forward_timing":timing_rows,
    "notes":["Archived test belongs to selection45k, not final50k",
             "Demo/subset/synthetic scores are not full-data benchmark evidence",
             "Full-data Adam/multi-LR results remain absent in the original PDF snapshot"]}
atomic_json(OUTPUT_ROOT/"presentation_receipt.json",receipt)
print("Output:",OUTPUT_ROOT)
print("Receipt:",OUTPUT_ROOT/"presentation_receipt.json")
display(Markdown("**Trình bày:** dữ liệu → block/forward → budget → loss/optimizer → một epoch/resume → kết quả 600 epoch và test đã lưu → giới hạn của bài làm."))